# 04 · Two-Tier Verification

**Tier 1 — BiLSTM-FiLM Hybrid Scorer**  
Detects anomalies using a weighted combination of Reconstruction error (α) and Forecasting error (β).  
Dynamic thresholds + z-score gating suppress false alarms before any LLM call is made.

**Tier 2 — GenAI Auditor**  
Only fires for windows that cleared the Tier-1 confirmed-anomaly gate AND exceed `TIER1_MIN_SCORE_RATIO`.  
Routes to **Google Gemini** (primary) with automatic **OpenAI GPT** fallback.  
Returns: `label · severity · root_cause · impact_analysis · recommended_action`.

---
```
Window → [Tier 1: Hybrid Score] → if confirmed_anomaly
               → [Context Builder] → [Tier 2: Gemini / GPT]
                                         → Final Decision
```

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(os.path.abspath("."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR  = (PROJECT_ROOT / "data"    / "research_processed_smoke_auto").resolve()
MODEL_DIR    = (PROJECT_ROOT / "models"  / "research_smoke_auto").resolve()
RESULTS_DIR  = (PROJECT_ROOT / "results" / "research_smoke_auto").resolve()

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"RESULTS_DIR   : {RESULTS_DIR}")

## 1 · Configuration

In [ ]:
from src.config import GenAIConfig, GPTConfig

# ── GenAI config (reads GENAI_PROVIDER / GEMINI_API_KEY / OPENAI_API_KEY from .env) ──
genai_cfg = GenAIConfig(
    output_dir=RESULTS_DIR,
    max_records=20,          # cap LLM calls during this notebook run
    tier1_min_score_ratio=1.05,
    tier2_enabled=True,
)

print(f"Provider       : {genai_cfg.provider}")
print(f"Gemini model   : {genai_cfg.gemini_model}")
print(f"Gemini key set : {bool(genai_cfg.gemini_api_key)}")
print(f"OpenAI key set : {bool(genai_cfg.openai_api_key)}")
print(f"Tier-1 min ratio : {genai_cfg.tier1_min_score_ratio}")
print(f"Tier-2 enabled   : {genai_cfg.tier2_enabled}")

## 2 · Load Tier-1 Candidate Windows

Load the confirmed anomaly candidates produced by the evaluation pipeline (notebook 03).  
These are **already scored by the BiLSTM-FiLM hybrid scorer** — no inference needed here.

In [ ]:
import pandas as pd

# Try successive fallback paths
candidate_paths = [
    RESULTS_DIR / "realtime_alert_candidates.csv",
    RESULTS_DIR / "hybrid" / "realtime_alert_candidates.csv",
    RESULTS_DIR / "hybrid" / "realtime_stream_predictions.csv",
    RESULTS_DIR / "realtime_stream_predictions.csv",
]

candidates_df = None
for p in candidate_paths:
    if p.exists():
        raw = pd.read_csv(p)
        # Filter to confirmed anomalies when loading from full stream file
        if "decision" in raw.columns:
            candidates_df = raw[raw["decision"] == "confirmed_anomaly"].copy()
        elif "is_candidate" in raw.columns:
            candidates_df = raw[raw["is_candidate"] == True].copy()
        else:
            candidates_df = raw.copy()
        print(f"Loaded {len(candidates_df)} candidates from: {p}")
        break

if candidates_df is None or candidates_df.empty:
    raise FileNotFoundError(
        "No candidate file found. Run notebook 03 first to generate evaluation outputs."
    )

# Show score distribution
score_col = "final_score" if "final_score" in candidates_df.columns else "anomaly_score"
candidates_df.sort_values(score_col, ascending=False, inplace=True)
candidates_df.reset_index(drop=True, inplace=True)

print(f"\nTier-1 score stats:")
print(candidates_df[score_col].describe().to_string())
candidates_df[["window_id", "container_id", score_col,
               "recon_score", "forecast_score",
               "decision_reason"]].head(10)

## 3 · Tier-1 Score Distribution Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

scores = candidates_df[score_col].values
thresholds = candidates_df["threshold"].values if "threshold" in candidates_df.columns else None

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left — histogram
ax = axes[0]
ax.hist(scores, bins=30, color="#e74c3c", alpha=0.85, edgecolor="white", linewidth=0.5)
ax.set_title("Tier-1 Hybrid Score Distribution (Confirmed Anomalies)", fontsize=11, fontweight="bold")
ax.set_xlabel("Hybrid Anomaly Score")
ax.set_ylabel("Window Count")
if thresholds is not None:
    ax.axvline(float(np.median(thresholds)), color="#2c3e50", linestyle="--", lw=1.5, label="median threshold")
    ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# Right — recon vs forecast scatter
ax2 = axes[1]
if {"recon_score", "forecast_score"}.issubset(candidates_df.columns):
    scatter = ax2.scatter(
        candidates_df["recon_score"], candidates_df["forecast_score"],
        c=scores, cmap="Reds", s=40, alpha=0.7, edgecolors="none"
    )
    plt.colorbar(scatter, ax=ax2, label="Hybrid Score")
    ax2.set_xlabel("Reconstruction Score (α)"); ax2.set_ylabel("Forecast Score (β)")
    ax2.set_title("Recon vs Forecast Scores", fontsize=11, fontweight="bold")
    ax2.grid(alpha=0.25)
else:
    ax2.text(0.5, 0.5, "recon_score / forecast_score\nnot available",
             ha="center", va="center", transform=ax2.transAxes, color="grey")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "tier1_score_distribution.png", dpi=150)
plt.show()

## 4 · Tier-2 GenAI Audit — Two-Tier Verifier

For each Tier-1 confirmed anomaly:
1. Build enriched context (metrics + metadata + top deviating features + score ratio)
2. Call **Gemini** (or **OpenAI** if Gemini key missing / call fails)
3. Receive structured JSON: `label · severity · root_cause · impact_analysis · recommended_action`

In [ ]:
from src.two_tier_verifier import TwoTierVerifier

verifier = TwoTierVerifier(config=genai_cfg)

# Cap to max_records to control API spend
records_to_audit = candidates_df.head(genai_cfg.max_records).to_dict(orient="records")
print(f"Running Two-Tier Verification on {len(records_to_audit)} candidates ...")

results = verifier.verify_batch(records_to_audit)

print(f"\nDone. {sum(r.tier2_triggered for r in results)} windows reached Tier-2.")
print(f"Providers used: { {r.tier2_provider for r in results if r.tier2_provider} }")

## 5 · Decision Comparison — Tier-1 Rule vs Tier-2 GenAI

In [ ]:
rows = [r.to_dict() for r in results]
results_df = pd.DataFrame(rows)

display_cols = [
    "window_id", "container_id",
    "tier1_score", "tier1_score_ratio", "tier1_flagged",
    "tier2_triggered", "tier2_provider", "tier2_latency_ms", "tier2_used_fallback",
    "final_label", "final_severity", "final_recommended_action",
]
display_cols = [c for c in display_cols if c in results_df.columns]
print(results_df[display_cols].to_string(index=False, max_colwidth=30))

## 6 · Label Distribution & False-Positive Filter Rate

In [ ]:
if not results_df.empty and "final_label" in results_df.columns:
    label_counts = results_df["final_label"].value_counts()
    fp_count = int(label_counts.get("normal", 0))
    total = len(results_df)
    tier2_fired = int(results_df["tier2_triggered"].sum())

    print("\n── Label Distribution ──")
    print(label_counts.to_string())
    print(f"\nTier-1 candidates       : {total}")
    print(f"Tier-2 triggered        : {tier2_fired}")
    print(f"False positives removed : {fp_count}  "
          f"({100*fp_count/max(tier2_fired,1):.1f}% of LLM-reviewed windows)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Label donut
    ax = axes[0]
    colors = {"normal": "#2ecc71", "warning": "#f39c12",
              "fault_candidate": "#e67e22", "critical": "#c0392b"}
    wedge_colors = [colors.get(l, "#95a5a6") for l in label_counts.index]
    wedges, texts, autotexts = ax.pie(
        label_counts.values, labels=label_counts.index,
        colors=wedge_colors, autopct="%1.0f%%",
        startangle=90, wedgeprops=dict(width=0.55)
    )
    ax.set_title("Tier-2 Label Distribution", fontweight="bold")

    # Severity bar
    ax2 = axes[1]
    if "final_severity" in results_df.columns:
        sev_counts = results_df["final_severity"].value_counts()
        sev_colors = {"low": "#27ae60", "medium": "#e67e22", "high": "#c0392b"}
        bars = ax2.bar(
            sev_counts.index, sev_counts.values,
            color=[sev_colors.get(s, "#95a5a6") for s in sev_counts.index],
            edgecolor="white"
        )
        ax2.bar_label(bars, padding=3, fontsize=10)
        ax2.set_title("Severity Breakdown", fontweight="bold")
        ax2.set_ylabel("Count")
        ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "tier2_label_distribution.png", dpi=150)
    plt.show()

## 7 · Root Cause & Recommendations — Top Critical Windows

In [ ]:
if not results_df.empty:
    critical_mask = results_df["final_label"].isin(["critical", "fault_candidate"])
    critical_df = results_df[critical_mask].copy()
    print(f"Critical / fault_candidate windows: {len(critical_df)}\n")

    for i, row in critical_df.head(5).iterrows():
        print(f"{'='*70}")
        print(f"Window {row['window_id']}  |  Container: {row['container_id']}")
        print(f"  Tier-1 score   : {row['tier1_score']:.4f}  "
              f"(ratio={row['tier1_score_ratio']:.2f}x threshold)")
        print(f"  Tier-2 provider: {row.get('tier2_provider', 'N/A')}  "
              f"({row.get('tier2_latency_ms', 'N/A')} ms)")
        print(f"  Final label    : {row['final_label']}  |  "
              f"Severity: {row.get('final_severity', 'N/A')}")
        print(f"  Root cause     : {row.get('final_root_cause', 'N/A')}")
        print(f"  Impact         : {row.get('final_impact_analysis', 'N/A')}")
        print(f"  Action         : {row.get('final_recommended_action', 'N/A')}")
        steps = row.get("final_step_by_step_recommendations", [])
        if steps:
            for j, step in enumerate(steps[:3], 1):
                print(f"    Step {j}: {step}")
        print()

## 8 · Latency Analysis

In [ ]:
if "tier2_latency_ms" in results_df.columns:
    lat = results_df[results_df["tier2_triggered"]]["tier2_latency_ms"].dropna()
    if not lat.empty:
        print("Tier-2 Latency (ms):")
        print(lat.describe().to_string())

        fig, ax = plt.subplots(figsize=(8, 3))
        ax.barh(range(len(lat)), sorted(lat.values), color="#3498db", edgecolor="white")
        ax.axvline(lat.mean(), color="#e74c3c", linestyle="--", lw=1.5, label=f"mean={lat.mean():.0f} ms")
        ax.set_xlabel("Latency (ms)")
        ax.set_title("Tier-2 GenAI Call Latency per Window", fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "tier2_latency.png", dpi=150)
        plt.show()
    else:
        print("No Tier-2 latency data available (Tier-2 may not have triggered).")

## 9 · Save Outputs

In [ ]:
from src.utils import ensure_directory, write_json

ensure_directory(RESULTS_DIR)

# Save full results dataframe
out_csv = RESULTS_DIR / "two_tier_verification_results.csv"
results_df.drop(columns=[c for c in ["tier2_context", "tier2_raw_decision"]
                          if c in results_df.columns], inplace=True)
results_df.to_csv(out_csv, index=False)
print(f"Saved results → {out_csv}")

# Save JSON with raw decisions
out_json = RESULTS_DIR / "two_tier_verification_results.json"
write_json(out_json, {"records": [r.to_dict() for r in results]})
print(f"Saved JSON    → {out_json}")

# Summary stats
summary = {
    "total_tier1_candidates": len(results),
    "tier2_triggered": int(sum(r.tier2_triggered for r in results)),
    "tier2_providers": list({r.tier2_provider for r in results if r.tier2_provider}),
    "false_positives_filtered": int(sum(
        1 for r in results if r.tier2_triggered and r.final_label == "normal"
    )),
    "label_distribution": results_df["final_label"].value_counts().to_dict()
        if "final_label" in results_df.columns else {},
    "severity_distribution": results_df["final_severity"].value_counts().to_dict()
        if "final_severity" in results_df.columns else {},
}
write_json(RESULTS_DIR / "two_tier_summary.json", summary)
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")